In [21]:
import datetime
import logging
import re
from pathlib import Path

import torch
import trackio
from datasets import load_dataset
from huggingface_hub.errors import GatedRepoError
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

In [22]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("gradio_client").setLevel(logging.WARNING)

In [23]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU available. This notebook requires a GPU for efficient training.")
    print("In Colab: Runtime → Change runtime type → Hardware accelerator → GPU")

CUDA available: True
Number of GPUs: 1
Current GPU: 0
GPU name: NVIDIA A40
GPU memory: 47.7 GB


## Configuration

In [24]:
_local_model_dir = Path("gemma-3-1b-it")
model_name = str(_local_model_dir) 
max_seq_length = 1024
max_prompt_length = 256
lora_rank = 32

print(f"model_name = {model_name}")

model_name = gemma-3-1b-it


In [25]:
# GSM8K structured reasoning format copied from the HF TRL GRPO cookbook.
reasoning_start = "<start_working_out>"
reasoning_end = "<end_working_out>"
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = f"""You are a mathematical reasoning assistant.
When given a math problem:
1. Show your step-by-step work between {reasoning_start} and {reasoning_end}
2. Provide your final numerical answer between {solution_start} and {solution_end}
3. Be precise and show all calculation steps clearly."""

print(system_prompt)

You are a mathematical reasoning assistant.
When given a math problem:
1. Show your step-by-step work between <start_working_out> and <end_working_out>
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>
3. Be precise and show all calculation steps clearly.


## Dataset: GSM8K

In [26]:
def extract_hash_answer(text):
    """Extract numerical answer from GSM8K format (#### marker)."""
    if "####" not in text:
        return None
    return text.split("####")[1].strip()


def process_dataset_example(example):
    """Convert GSM8K example to conversation format for GRPO training."""
    question = example["question"]
    answer = extract_hash_answer(example["answer"])
    prompt = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    return {
        "prompt": prompt,
        "answer": answer,
    }


gsm8k_local_dir = Path("/workspace/gsm8k-grade-school-math-8k-dataset/gsm8k/main")


def load_gsm8k_dataset():
    print("Loading GSM8K mathematical reasoning dataset...")
    dataset = load_dataset(
        "parquet",
        data_files={"train": str(gsm8k_local_dir / "train-00000-of-00001.parquet")},
        split="train",
    )
    dataset = dataset.map(process_dataset_example)

    assert "prompt" in dataset.column_names
    assert "answer" in dataset.column_names
    assert len(dataset[0]["prompt"]) == 2
    assert dataset[0]["answer"] is not None

    print("Dataset loaded and processed.")
    print(f"Training examples: {len(dataset):,}")
    print(f"Sample row: {dataset[0]}")
    print("-------------------------")
    print(f"Sample question: {dataset[0]['prompt'][1]['content']}...")
    print(f"Sample answer: {dataset[0]['answer']}")
    return dataset

In [27]:
dataset = load_gsm8k_dataset()
dataset[0]

Loading GSM8K mathematical reasoning dataset...
Dataset loaded and processed.
Training examples: 7,473
Sample row: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': '72', 'prompt': [{'role': 'system', 'content': 'You are a mathematical reasoning assistant.\nWhen given a math problem:\n1. Show your step-by-step work between <start_working_out> and <end_working_out>\n2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>\n3. Be precise and show all calculation steps clearly.'}, {'role': 'user', 'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}]}
-------------------------
Sample question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogethe

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'role': 'system',
   'content': 'You are a mathematical reasoning assistant.\nWhen given a math problem:\n1. Show your step-by-step work between <start_working_out> and <end_working_out>\n2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>\n3. Be precise and show all calculation steps clearly.'},
  {'role': 'user',
   'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}]}

## Prompt length check

Sanity-check `max_prompt_length` against the actual tokenized prompt lengths for this dataset/tokenizer, since it's an assumed value, not a derived one. Note this trl version's `GRPOConfig` has no `max_prompt_length` field, so prompts are never truncated - this only affects the `max_completion_length` budget (`max_seq_length - max_prompt_length`).

In [28]:
_length_check_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

_prompt_token_lengths = sorted(
    len(
        _length_check_tokenizer(
            _length_check_tokenizer.apply_chat_template(
                example["prompt"], add_generation_prompt=True, tokenize=False
            )
        )["input_ids"]
    )
    for example in dataset
)

_n = len(_prompt_token_lengths)
_over_budget = sum(1 for length in _prompt_token_lengths if length > max_prompt_length)

print(f"Prompts checked: {_n:,}")
print(f"min={_prompt_token_lengths[0]}  "
      f"p50={_prompt_token_lengths[_n // 2]}  "
      f"p90={_prompt_token_lengths[int(_n * 0.9)]}  "
      f"p95={_prompt_token_lengths[int(_n * 0.95)]}  "
      f"p99={_prompt_token_lengths[int(_n * 0.99)]}  "
      f"max={_prompt_token_lengths[-1]}")
print(f"max_prompt_length = {max_prompt_length}")
print(f"Prompts exceeding max_prompt_length: {_over_budget} ({_over_budget / _n:.2%})")
if _over_budget:
    print(
        "These won't be truncated (this trl version's GRPOConfig has no "
        "max_prompt_length field), but they'll eat into the max_completion_length budget."
    )

Prompts checked: 7,473
min=92  p50=135  p90=168  p95=180  p99=207  max=293
max_prompt_length = 256
Prompts exceeding max_prompt_length: 4 (0.05%)
These won't be truncated (this trl version's GRPOConfig has no max_prompt_length field), but they'll eat into the max_completion_length budget.


### Completion length budget

tokenize the *reference* GSM8K solutions wrapped in our target format to see how long a **correct** completion actually needs to be. A cap below the max clips legitimate answers

In [29]:
_reference_answers = load_dataset(
    "parquet",
    data_files={"train": str(gsm8k_local_dir / "train-00000-of-00001.parquet")},
    split="train",
)["answer"]

def _format_reference_completion(answer_field):
    _reasoning, _final = answer_field.split("####")
    _reasoning = re.sub(r"<<.*?>>", "", _reasoning).strip()  # drop GSM8K calculator annotations
    return f"{reasoning_start}\n{_reasoning}\n{reasoning_end}\n{solution_start}{_final.strip()}{solution_end}"

_completion_token_lengths = sorted(
    len(_length_check_tokenizer(_format_reference_completion(ans))["input_ids"])
    for ans in _reference_answers
)

_m = len(_completion_token_lengths)
def _cpct(p):
    return _completion_token_lengths[min(int(_m * p), _m - 1)]

print(f"Reference completions checked: {_m:,}")
print(f"min={_completion_token_lengths[0]}  "
      f"p50={_cpct(0.5)}  p90={_cpct(0.9)}  p95={_cpct(0.95)}  "
      f"p99={_cpct(0.99)}  max={_completion_token_lengths[-1]}")
print("Coverage by max_completion_length cap:")
for _cap in [256, 384, 512, 640, 768]:
    _covered = sum(1 for length in _completion_token_lengths if length <= _cap) / _m
    print(f"  cap={_cap:4d} -> covers {_covered:6.1%} of reference solutions")

Reference completions checked: 7,473
min=44  p50=104  p90=169  p95=194  p99=241  max=364
Coverage by max_completion_length cap:
  cap= 256 -> covers  99.4% of reference solutions
  cap= 384 -> covers 100.0% of reference solutions
  cap= 512 -> covers 100.0% of reference solutions
  cap= 640 -> covers 100.0% of reference solutions
  cap= 768 -> covers 100.0% of reference solutions


## Reward functions

Four reward functions (ported from the TRL GRPO reasoning cookbook's GSM8K recipe): exact format match, approximate/partial format credit, numeric correctness with partial credit, and raw number extraction.

In [30]:
match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{reasoning_start}.+?{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)
match_numbers = re.compile(
    rf"{solution_start}.*?([\d\.]{{1,}})",
    flags=re.MULTILINE | re.DOTALL,
)

In [31]:
def match_format_exactly(completions, **kwargs):
    """High reward for perfect format adherence."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        score = 3.0 if match_format.search(response) is not None else 0.0
        scores.append(score)
    return scores

In [32]:
def match_format_approximately(completions, **kwargs):
    """Graduated scoring for format elements."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        score = 0
        score += 0.5 if response.count(reasoning_start) == 1 else -0.5
        score += 0.5 if response.count(reasoning_end) == 1 else -0.5
        score += 0.5 if response.count(solution_start) == 1 else -0.5
        score += 0.5 if response.count(solution_end) == 1 else -0.5
        scores.append(score)
    return scores

In [33]:
def check_answer_correctness(prompts, completions, answer, **kwargs):
    """Graduated scoring for mathematical accuracy."""
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(0)
            continue
        if guess.strip() == true_answer.strip():
            scores.append(3.0)
        else:
            try:
                ratio = float(guess) / float(true_answer)
                if 0.9 <= ratio <= 1.1:
                    scores.append(1.5)
                elif 0.8 <= ratio <= 1.2:
                    scores.append(0.5)
                else:
                    scores.append(-0.5)
            except (ValueError, ZeroDivisionError):
                scores.append(-0.5)
    return scores

In [34]:
def check_numbers_extraction(prompts, completions, answer, **kwargs):
    """Tests the model's ability to extract numerical values from solution sections."""
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [
        guess.group(1) if (guess := match_numbers.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(0)
            continue
        try:
            true_val = float(true_answer.strip())
            guess_val = float(guess.strip())
            scores.append(1.5 if guess_val == true_val else 0.0)
        except (ValueError, TypeError):
            scores.append(0)
    return scores

## Model + tokenizer (bf16 LoRA)

In [35]:
def load_model_and_tokenizer():
    print(f"Loading model: {model_name}")
    print(f"Max sequence length: {max_seq_length}")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
        # attn_implementation="flash_attention_2",
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_rank,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_dropout=0.1,
        task_type=TaskType.CAUSAL_LM,
    )
    print("Applying LoRA adaptation to model...")
    model = get_peft_model(model, lora_config)
    print("LoRA Training Parameters Summary:")
    model.print_trainable_parameters()

    return model, tokenizer

In [36]:
model, tokenizer = load_model_and_tokenizer()

Loading model: gemma-3-1b-it
Max sequence length: 1024


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Applying LoRA adaptation to model...
LoRA Training Parameters Summary:
trainable params: 26,091,520 || all params: 1,025,977,472 || trainable%: 2.5431


In [37]:
!nvidia-smi

Sun Jul  5 20:08:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:CE:00.0 Off |                    0 |
|  0%   41C    P0            107W /  300W |    2353MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## GRPO training configuration

In [ ]:
def build_training_args():
    return GRPOConfig(
        learning_rate=5e-6,
        max_grad_norm=0.1,
        bf16=True,

        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_generations=8,
        # max_prompt_length=max_prompt_length,
        max_completion_length=786,
        # max_completion_length=max_seq_length - max_prompt_length, #768
        # mask_truncated_completions=True,

        num_train_epochs=1,
        save_steps=250,
        logging_steps=1,

        output_dir="./gemma3_1b_gsm8k_grpo_outputs",
        report_to="trackio",
        log_on_each_node=False,
    )

### GRPO config with vLLM rollout (colocate)

In [ ]:
def build_training_args_vllm():
    return GRPOConfig(
        learning_rate=5e-6,
        max_grad_norm=0.1,
        bf16=True,

        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_generations=8,
        max_completion_length=768,

        num_train_epochs=1,
        save_steps=250,
        logging_steps=1,

        use_vllm=True,
        vllm_mode="colocate",
        vllm_gpu_memory_utilization=0.3, 
        vllm_max_model_length=1280,
        # vllm_tensor_parallel_size=1,
        vllm_enable_sleep_mode=False,

        output_dir="./gemma3_1b_gsm8k_grpo_outputs",
        report_to="trackio",
        log_on_each_node=False,
    )

# generation_batch_size is auto calculated based on generation_batch_size = per_device_train_batch_size × num_processes × steps_per_generation


training_args = build_training_args_vllm()
print("vLLM rollout enabled :", training_args.use_vllm)
print("vLLM mode            :", training_args.vllm_mode)
print("vLLM GPU mem fraction:", training_args.vllm_gpu_memory_utilization)
print("generation_batch_size:", training_args.generation_batch_size)

vLLM rollout enabled : True
vLLM mode            : colocate
vLLM GPU mem fraction: 0.3
generation_batch_size: 16


## Experiment tracking (trackio)

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
run_name = f"simple-reward-{timestamp}"
trackio.init(
    project="GRPO-Mathematical-Reasoning",
    name=run_name,
    config={
        "model_name": model_name,
        "dataset": "GSM8K",
        "technique": "GRPO + LoRA (bf16)",
        "learning_rate": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "effective_batch_size": (
            training_args.per_device_train_batch_size
            * training_args.gradient_accumulation_steps
        ),
        "max_steps": training_args.max_steps,
        "lora_r": lora_rank,
        "lora_alpha": lora_rank,
        "num_generations": training_args.num_generations,
        "max_prompt_length": max_prompt_length,
        "max_completion_length": training_args.max_completion_length,
        "num_reward_functions": 4,
        "mask_truncated_completions": training_args.mask_truncated_completions,
    },
)
print(f"Run name: {run_name}")

## Build GRPOTrainer and train

In [40]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        check_answer_correctness,
        check_numbers_extraction,
    ],
    args=training_args,
    train_dataset=dataset,
)
print("GRPO Trainer initialized successfully.")
print(f"Training dataset: {len(dataset):,} examples")
print(f"Reward functions: {len(trainer.reward_funcs)} active")

/usr/local/lib/python3.11/dist-packages/trl/generation/vllm_generation.py:294: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.19.0. You have version 0.21.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if not is_vllm_available():


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 7/7 [00:00<00:00, 29.43it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 5/5 [00:00<00:00, 43.78it/s]


GRPO Trainer initialized successfully.
Training dataset: 7,473 examples
Reward functions: 4 active


In [ ]:
print("Starting GRPO training...")
trainer.train()

In [ ]:
try:
    trackio.finish()
except RuntimeError as exc:
    print(f"Trackio finish skipped: {exc}")

print("Training completed successfully.")
print(f"Model saved to: {training_args.output_dir}")

## Save model

In [ ]:
model.save_pretrained("grpo_saved_lora")
tokenizer.save_pretrained("grpo_saved_lora")

## Qualitative evaluation

In [ ]:
def test_model(model, tokenizer, question, max_length=512):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"Processing: {question}")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            length_penalty=1.0,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = response[len(text) :].strip()
    return generated_text

In [ ]:
gsm8k_question = (
    "Natalia sold clips to 48 of her friends in April, and then she sold "
    "half as many clips in May. How many clips did Natalia sell altogether "
    "in April and May?"
)
expected_answer = "72"
gsm8k_response = test_model(model, tokenizer, gsm8k_question, max_length=768)

print(f"Question: {gsm8k_question}")
print(f"Model Response:\n{gsm8k_response}")

has_reasoning = reasoning_start in gsm8k_response and reasoning_end in gsm8k_response
has_solution = solution_start in gsm8k_response and solution_end in gsm8k_response
print("\nFormat Check:")
print(f"Reasoning section: {has_reasoning}")
print(f"Solution section: {has_solution}")
if has_solution:
    try:
        solution_text = (
            gsm8k_response.split(solution_start)[1]
            .split(solution_end)[0]
            .strip()
        )
        extracted_number = "".join(filter(str.isdigit, solution_text))
        expected_number = "".join(filter(str.isdigit, expected_answer))
        is_correct = extracted_number == expected_number
        print(f"Extracted: {solution_text}")
        print(f"Expected: {expected_answer}")
        print(f"Correct: {is_correct}")
    except IndexError:
        print("Could not extract solution")